In [11]:
# Python Library
import os
import glob
import sys
import numpy as np

from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
from astropy.table import hstack
import warnings
warnings.filterwarnings("ignore")

# Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl

# Jupyter Setting
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [12]:
### Helper Functions
import sys
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *

In [13]:
logtxt = ""

In [14]:
input_rubin_table = os.path.join(AUGMENTED_BALANCED_PHOT_RUBIN_DETECTED_DATA, "synphot_normal_class_detection.csv")	
input_7dt_table = os.path.join(AUGMENTED_BALANCED_PHOT_7DT_DETECTED_DATA, "synphot_normal_class_detection.csv")	

rubin_table = Table.read(input_rubin_table)
sdt_table = Table.read(input_7dt_table)

print(f"Rubin Table: {len(rubin_table):,} rows")
print(f"7DT Table: {len(sdt_table):,} rows")

logtxt += f"Rubin Table: {len(rubin_table):,} rows\n"
logtxt += f"7DT Table: {len(sdt_table):,} rows\n"

Rubin Table: 294,235 rows
7DT Table: 292,491 rows


In [15]:
import pandas as pd
from astropy.table import Table

# 1) Astropy → pandas
df_rubin = rubin_table.to_pandas().reset_index().rename(columns={'index':'rubin_idx'})
df_sdt   = sdt_table.to_pandas().reset_index().rename(columns={'index':'sdt_idx'})

# 2) 'spec'과 'scale_factor'만 기준으로 inner join (공통 쌍만 남기기)
common = pd.merge(
    df_sdt[['spec','scale_factor']], 
    df_rubin[['spec','scale_factor']],
    on=['spec','scale_factor'],
    how='inner'
)

# 3) 이 공통 키를 기준으로 원본 테이블에서 필터링
df_rubin_common = pd.merge(
    df_rubin,
    common,
    on=['spec','scale_factor'],
    how='inner'
)
df_sdt_common = pd.merge(
    df_sdt,
    common,
    on=['spec','scale_factor'],
    how='inner'
)

# 4) 필요하면 다시 Astropy Table로!
rubin_common_table = Table.from_pandas(df_rubin_common.drop(columns=['rubin_idx']))
sdt_common_table   = Table.from_pandas(df_sdt_common  .drop(columns=['sdt_idx']))

logtxt += f"{len(rubin_common_table):,} rows\n"
logtxt += f"{len(sdt_common_table):,} rows\n"

In [16]:

logtxt += f"Number Check: {len(rubin_common_table):,} == {len(sdt_common_table):,} ({len(rubin_common_table) == len(sdt_common_table)})\n"
len(rubin_common_table) == len(sdt_common_table)

True

In [17]:
keys_common = list(set(rubin_common_table.keys()) & set(sdt_common_table.keys()))
key_ignore = [f"{key}_{i}" for key in keys_common for i in (1, 2)] + [
    key for key in sdt_common_table.keys() if any(x in key for x in ("n_iter", "scale_factor", "fnu", "magapp", "magerr", "snr", "magabs"))
] + [
    key for key in rubin_common_table.keys() if any(x in key for x in ("n_iter", "scale_factor", "fnu", "magapp", "magerr", "snr", "magabs"))
]

print(keys_common)
print(key_ignore)

['type', 'detection', 'spec', 'flagtype', 'n_iter', 'scale_factor']
['type_1', 'type_2', 'detection_1', 'detection_2', 'spec_1', 'spec_2', 'flagtype_1', 'flagtype_2', 'n_iter_1', 'n_iter_2', 'scale_factor_1', 'scale_factor_2', 'magabs_m400', 'snr_m400', 'magapp_m400', 'magerr_m400', 'fnuobs_m400', 'fnuerr_m400', 'fnu_m400', 'n_iter_m400', 'scale_factor_m400', 'magabs_m412', 'snr_m412', 'magapp_m412', 'magerr_m412', 'fnuobs_m412', 'fnuerr_m412', 'fnu_m412', 'n_iter_m412', 'scale_factor_m412', 'magabs_m425', 'snr_m425', 'magapp_m425', 'magerr_m425', 'fnuobs_m425', 'fnuerr_m425', 'fnu_m425', 'n_iter_m425', 'scale_factor_m425', 'magabs_m437', 'snr_m437', 'magapp_m437', 'magerr_m437', 'fnuobs_m437', 'fnuerr_m437', 'fnu_m437', 'n_iter_m437', 'scale_factor_m437', 'magabs_m450', 'snr_m450', 'magapp_m450', 'magerr_m450', 'fnuobs_m450', 'fnuerr_m450', 'fnu_m450', 'n_iter_m450', 'scale_factor_m450', 'magabs_m462', 'snr_m462', 'magapp_m462', 'magerr_m462', 'fnuobs_m462', 'fnuerr_m462', 'fnu_m462',

In [18]:
rubin_keys_to_merge = keys_common + [key for key in rubin_common_table.keys() if key not in keys_common and key not in key_ignore]
sdt_keys_to_merge = [key for key in sdt_common_table.keys() if key not in keys_common and key not in key_ignore]

merged_table = hstack([rubin_common_table[rubin_keys_to_merge], sdt_common_table[sdt_keys_to_merge]])
logtxt += f"Number of merged synphot tables: {len(merged_table):,}\n"
merged_table[:3]

type,detection,spec,flagtype,n_iter,scale_factor,magobs_u,magobs_g,magobs_r,magobs_i,magobs_z,magobs_y,magobs_m400,magobs_m412,magobs_m425,magobs_m437,magobs_m450,magobs_m462,magobs_m475,magobs_m487,magobs_m500,magobs_m512,magobs_m525,magobs_m537,magobs_m550,magobs_m562,magobs_m575,magobs_m587,magobs_m600,magobs_m612,magobs_m625,magobs_m637,magobs_m650,magobs_m662,magobs_m675,magobs_m687,magobs_m700,magobs_m712,magobs_m725,magobs_m737,magobs_m750,magobs_m762,magobs_m775,magobs_m787,magobs_m800,magobs_m812,magobs_m825,magobs_m837,magobs_m850,magobs_m862,magobs_m875,magobs_m887
str7,str4,str160,str12,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
Ia,True,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,aug_blue_red,0,3.53111691382141,16.613830322779968,16.077696049126388,15.980868060968696,16.17826474173846,16.297289171707153,16.27704317309181,16.237262644016845,15.961400231503017,15.962086940314064,16.194882711744796,16.182302776219927,15.959699065773934,15.942182049474008,16.108268432628698,16.269804743672715,16.190563754154272,15.965619528120387,16.049701763055037,16.198438457637295,16.04358212209503,15.912337127675816,15.927552879656647,15.94152861902329,16.023475962258846,16.403579933725055,16.19186228770287,15.766704395380764,15.762967899739884,15.86873813179745,16.01915088937519,16.064041335817937,16.111508326677367,16.161571821004788,16.10987227223634,16.19382042497373,16.37064015878588,16.317811874589477,16.17044788304539,16.168682616905283,16.333099089242857,16.364590976684507,16.405536130825435,16.443218320496147,16.25941758426884,16.101060066472396,16.07916381174072
Ia,True,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,aug_blue_red,0,0.7546696410796927,18.285813585231395,17.75304520694589,17.655889277067384,17.853327692367003,17.97163766333954,17.95679068928053,17.92377905937501,17.652160048425234,17.667649306965956,17.8449670503278,17.845233919267603,17.633408834366204,17.616371586173244,17.789301206585513,17.942055682869356,17.865029059974113,17.659056213702748,17.697784817281462,17.833729448699913,17.723436987607272,17.571192332148613,17.58786942088109,17.59980900862765,17.69609936163808,18.1025407843818,17.84530690191782,17.42224512083261,17.406472381843194,17.57689915326171,17.704099388595733,17.722012881300408,17.80193115740833,17.82091129502488,17.780999713374996,17.858420923143065,18.000919099091508,17.98690297831608,17.716971455402476,17.768974197425994,18.080292835048166,18.17643403972527,18.06693999815441,18.068217783923565,17.821982225697784,17.588539179971637,17.74655976904662
Ia,True,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/tPS1-15p_20150114_Gr13_Free_slit1.0_1_f.asci,aug_blue_red,0,5.214297905300546,16.189815279496923,15.654446177811092,15.557492239705477,15.7551622109708,15.874503356565526,15.852528028629003,15.812896232798881,15.52855843810268,15.559451388761616,15.78570234129938,15.76442933979215,15.53700984262908,15.527351057978594,15.699921928391754,15.85090607575371,15.764160839077492,15.540064431769348,15.619278232405703,15.749235237051613,15.624590386350965,15.4822517470143,15.499485860367692,15.518212639402666,15.596911322034297,15.983579976089668,15.763822457019529,15.350644065281964,15.3398945248969,15.480361255369061,15.609722275691176,15.670492503938666,15.67282370727715,15.714881241022022,15.685588052946477,15.728811231896428,15.9253768136962,15.872712327191863,15.738738519946397,15.735875127393003,15.897783813470765,15.997934406514844,15.964981777085898,16.036165133277716,15.82947578610233,15.67661492

In [19]:
merged_table_name = os.path.join(AUGMENTED_BALANCED_PHOT_DATA, "merged_synphot_normal_class.csv")
merged_table.write(merged_table_name, format='csv', overwrite=True)
print(f"Merged table saved to {merged_table_name}")
logtxt += f"Merged table saved to {merged_table_name}\n"
logtxt += "\n"
logtxt += "END\n"
print(logtxt)

Merged table saved to /Users/paek/Research/7DT/SED-Classifier/notebook/../data/Synphot/Augmented_Balanced_Photometry/merged_synphot_normal_class.csv
Rubin Table: 294,235 rows
7DT Table: 292,491 rows
282,753 rows
282,753 rows
Number Check: 282,753 == 282,753 (True)
Number of merged synphot tables: 282,753
Merged table saved to /Users/paek/Research/7DT/SED-Classifier/notebook/../data/Synphot/Augmented_Balanced_Photometry/merged_synphot_normal_class.csv

END



In [20]:
logtxt_path = os.path.join(AUGMENTED_BALANCED_PHOT_DATA, "log.txt")
with open(logtxt_path, 'w') as f:
    f.write(logtxt)